In [30]:
import formulallm.formula as f

import io
import contextlib

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama
from llama_index.core import PromptTemplate

In [4]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [5]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    code = f.load(filePath)
    return code

load_tool = FunctionTool.from_defaults(fn=load)

In [6]:
def solve():
    """Try to solve the partial model based on the domain constraints"""
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.solve("pm","1","Mapping.conforms")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

solve_tool = FunctionTool.from_defaults(fn=solve)

In [7]:
def extract(task_id: str):
    """Extract the solve result to the task with task_id.
    If the partial model is solvable, will return the solution.
    Else if the partial model is unsolvable, will return the least unsatisfied core conditions.
    The task_id is a string representing an integer which is incremented by 1 each time we run the solve command."""
    
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.extract(task_id, "0", "0")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [8]:
react_agent = ReActAgent.from_tools(tools=[load_tool, solve_tool, extract_result_tool], llm=llama3, verbose=True)

In [28]:
react_system_header_str = """\

You are designed to help with a variety of tasks, from answering questions \
to providing summaries to other types of analyses.

## Tools
You have access to a wide variety of tools. You are responsible for using
the tools in any sequence you deem appropriate to complete the task at hand.
This may require breaking the task into subtasks and using different tools
to complete each subtask.

You have access to the following tools:
{tool_desc}

## Output Format
You should choose the appropriate format based on the task at hand.

### If you need to use a tool:
```
Thought: I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names})
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

The user will respond with the tool's output in the following format:
```
Observation: tool response
```

You should keep repeating the above format until you have enough information
to answer the question without using any more tools. At that point, you MUST respond
in one of the following two formats:
```
Thought: I can answer without using any more tools.
Answer: [your answer here]
```

```
Thought: I cannot answer the question with the provided tools.
Answer: Sorry, I cannot answer your query.
```

### If you do not need to use a tool and can rely on your NLP abilities:
```
Thought: Provide your reasoning or analysis here.
Answer: Provide your final answer here.
```

Please ensure that your responses are clear and concise, following the structure provided above.

## Current Conversation
Below is the current conversation consisting of interleaving human and assistant messages.

"""


In [31]:
react_system_prompt = PromptTemplate(react_system_header_str)
react_agent.update_prompts({"agent_worker:system_prompt": react_system_prompt})

In [32]:
prompt = """
Load the file from file path './data/MappingExample.4ml'.
Then solve the partial model call the extract tool.
If the partial model is unsolvable, the extract tool will return the unsat core terms.
Use your own natural language processing ability to explain why it is unsolvable based on the file content and the unsatisfiable core conditions.
You shouldn't need any tools for the explanation step!
Lastly, suggest a fix to the invalidUtilization constraint to solve the conflict and make the partial model solvable.
You shouldn't need any tools for the fixing step!
"""

In [33]:
response = react_agent.chat(prompt)

print(str(response))

> Running step 8d0d710c-82b4-40d1-adbc-09307e24edd8. Step input: 
Load the file from file path './data/MappingExample.4ml'.
Then solve the partial model call the extract tool.
If the partial model is unsolvable, the extract tool will return the unsat core terms.
Use your own natural language processing ability to explain why it is unsolvable based on the file content and the unsatisfiable core conditions.
You shouldn't need any tools for the explanation step!
Lastly, suggest a fix to the invalidUtilization constraint to solve the conflict and make the partial model solvable.
You shouldn't need any tools for the fixing step!

Thought: I need to use a tool to help me answer the question.
Action: load
Action Input: {'filePath': './data/MappingExample.4ml'}
(Compiled) MappingExample.4ml
0.06s.
Observation: domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be 